# Day 22: Building a Production-Style RAG Service

A demo works once, on a good day, with a query you picked yourself. A service has to work every time, on a bad day, with input someone else typed in a hurry. This notebook builds a single RagService class that wires together validation, caching, retrieval, ranking, generation, attribution, and logging into one entry point: query().

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from config import Config
good = Config()
good.validate()
print('Valid config:', good)
try:
    Config(top_k=0).validate()
except ValueError as e:
    print('Invalid config correctly rejected:', e)


## Run the service

One call to query() runs validate, cache check, preprocess, retrieve, rank, generate, attribute, and log, in that order.

In [ ]:
from rag_service import RagService
service = RagService(Config())
response = service.query('What is RAG?')
print('status:', response.status)
print('sources:', response.sources)
print('cache_hit:', response.cache_hit)
print('timings_ms:', response.timings_ms)

repeat_response = service.query('What is RAG?')
print('\nrepeat cache_hit:', repeat_response.cache_hit)


## Logging output

Every request is logged as a single JSON line: enough to debug a real problem, a short query preview instead of the full text, never a raw stack trace.

In [ ]:
import service_logging
service_logging.main()


## Load test results

p50 and p95 latency matter more than an average, since an average can hide a painful tail of slow requests. Caching on vs off, side by side.

In [ ]:
import load_test
load_test.main()


## Failure drills

Four failures practiced on purpose: an empty index, an injected exception inside the generator, an oversized query, and a degraded health state.

In [ ]:
import failure_drills
failure_drills.main()


## Health and metrics

Request count, error count, cache hit rate, and average latency per stage, aggregated across many requests, answering a different question than any single request's timing can.

In [ ]:
import health_and_metrics
health_and_metrics.main()


## Evaluation hook

Reuses Day 8's real precision, recall, and NDCG metrics, pointed at this service's own retriever, comparing top_k=3 against top_k=1.

In [ ]:
import evaluation_hook
evaluation_hook.main()


## Key takeaways

- A production service adds validation, logging, timing, caching, and quality measurement around the same retrieval and generation logic from earlier days, it does not replace any of it.
- Failing early and cheaply on a bad request means expensive stages never run on input that was never going to succeed.
- Every stage is timed separately, so a slow request can be traced to the exact stage responsible.
- Exceptions are caught and turned into a safe message, the real error goes to the log instead of to the user.
- Percentiles reveal a slow tail that an average would hide.
- Testing failures on purpose, before users find them, is what makes the safe-error behavior trustworthy instead of just hoped for.
- Quality has to be measured with real metrics on a fixed labeled set, not assumed from the service simply running without errors.